<a href="https://colab.research.google.com/github/haniyawahabb/My-QloRA-Project/blob/main/My_QloRA_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Thu Oct  1 10:40:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q -U transformers datasets peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 110.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 62.4 MB/s eta 0:00:00


In [ ]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig
from trl import SFTTrainer

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU: Tesla T4


In [ ]:
data = [
    {
        "instruction": "What is Python?",
        "response": "Python is a programming language used for web development, data science, artificial intelligence, and machine learning."
    },
    {
        "instruction": "What is Artificial Intelligence?",
        "response": "Artificial Intelligence is technology that enables computers to perform tasks that normally require human intelligence."
    },
    {
        "instruction": "What is Machine Learning?",
        "response": "Machine Learning is a branch of AI where computers learn patterns from data and use those patterns to make predictions."
    },
    {
        "instruction": "What is Deep Learning?",
        "response": "Deep Learning is a type of machine learning that uses neural networks with multiple layers."
    },
    {
        "instruction": "What is an LLM?",
        "response": "LLM stands for Large Language Model. It is an AI model trained on large amounts of text to understand and generate language."
    },
    {
        "instruction": "What is LoRA?",
        "response": "LoRA stands for Low-Rank Adaptation. It is a technique used to efficiently fine-tune large language models."
    },
    {
        "instruction": "What is QLoRA?",
        "response": "QLoRA combines LoRA with 4-bit quantization to reduce GPU memory usage during fine-tuning."
    },
    {
        "instruction": "What is RAG?",
        "response": "RAG stands for Retrieval-Augmented Generation. It retrieves relevant information before generating an answer."
    },
    {
        "instruction": "What is a chatbot?",
        "response": "A chatbot is a software application that communicates with users using natural language."
    },
    {
        "instruction": "What is an AI Engineer?",
        "response": "An AI Engineer develops and deploys artificial intelligence and machine learning systems."
    }
]

dataset = Dataset.from_list(data)

print(dataset)

Dataset({
    features: ['instruction', 'response'],
    num_rows: 10
})


In [ ]:
def format_data(example):
    return {
        "text": f"""### Instruction:
{example["instruction"]}

### Response:
{example["response"]}"""
    }

dataset = dataset.map(format_data)

print(dataset[0]["text"])

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

### Instruction:
What is Python?

### Response:
Python is a programming language used for web development, data science, artificial intelligence, and machine learning.


In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Base Model:", model_name)

Base Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded successfully!")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Tokenizer loaded successfully!


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("4-bit QLoRA configuration ready!")

4-bit QLoRA configuration ready!


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model.config.use_cache = False

print("4-bit model loaded successfully!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

4-bit model loaded successfully!


In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

print("LoRA configuration ready!")

LoRA configuration ready!


In [ ]:
training_args = TrainingArguments(
    output_dir="./qlora_output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    fp16=False, # Changed to False to avoid BFloat16 issues on T4
    report_to="none"
)

print("Training configuration ready!")

Training configuration ready!


In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
    args=training_args
)

print("Trainer created successfully!")

Adding EOS to train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Trainer created successfully!


In [ ]:
trainer.train()

Step,Training Loss
1,2.503195
2,2.197705
3,2.112545
4,1.612942
5,1.823902
6,1.646574


TrainOutput(global_step=6, training_loss=1.9828104972839355, metrics={'train_runtime': 5.792, 'train_samples_per_second': 5.18, 'train_steps_per_second': 1.036, 'total_flos': 8104340275200.0, 'train_loss': 1.9828104972839355, 'epoch': 3.0})

In [ ]:
output_dir = "./qlora_adapter"

trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print("QLoRA adapter saved successfully!")
print("Saved at:", output_dir)

QLoRA adapter saved successfully!
Saved at: ./qlora_adapter


In [ ]:
prompt = """### Instruction:
What is QLoRA?

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(answer)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
What is QLoRA?

### Response:
I don't know what QLoRA is.


In [ ]:
questions = [
    "What is Python?",
    "What is Machine Learning?",
    "What is LoRA?",
    "What is QLoRA?",
    "What is RAG?"
]

for question in questions:
    prompt = f"""### Instruction:
{question}

### Response:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("=" * 60)
    print(answer)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
What is Python?

### Response:
Python is a programming language.


[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
What is Machine Learning?

### Response:
Machine Learning is the process of using algorithms to learn from data.


[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
What is LoRA?

### Response:
LoRA is a shortened version of Long Range Aperture Radar.


[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
What is QLoRA?

### Response:
QLoRA is a program that helps students learn and practice math concepts.
### Instruction:
What is RAG?

### Response:
Rag is a type of protein that is involved in the immune response.
